In [ ]:
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf

import matplotlib.pyplot as plt

# Download the 'malaria' dataset

In [ ]:
ds_train, ds_info = tfds.load(
    'malaria',
    split='train',
    with_info=True,
    as_supervised=True,
)

In [ ]:
tfds.visualization.show_examples(
    ds=ds_train,
    ds_info=ds_info,
    is_batched=False,
)

In [ ]:
class_name = ['parasitized', 'uninfected']

In [ ]:
ds_train_ = ds_train.shuffle(buffer_size=10000, reshuffle_each_iteration=False)
num_train_examples = int(0.7 * len(ds_train))
ds_train = ds_train_.take(num_train_examples)
ds_test = ds_train_.skip(num_train_examples)

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 64

In [ ]:
def preprocess_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.GAUSSIAN)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, tf.one_hot(label, depth=2)

In [ ]:
ds_train = ds_train.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
ds_train = ds_train.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_train.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image[0].numpy())
    plt.title(class_name[np.argmax(label[0])])
    plt.axis("off")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Conv2D, Activation, BatchNormalization, MaxPooling2D, Dropout
from tensorflow.keras import regularizers, optimizers

In [ ]:
def get_model(base_hidden_units=32, weight_decay=1e-4):
    md = Sequential()
    
    md.add(Input(shape=(IMG_SIZE, IMG_SIZE, 3)))

    # CONV_1
    md.add(Conv2D(32, kernel_size=(3, 3), strides=(1, 1), padding='same',
                  kernel_regularizer=regularizers.l2(weight_decay),
                  ))
    md.add(Activation('relu'))
    md.add(BatchNormalization())

    # CONV_2
    md.add(Conv2D(32, kernel_size=(3, 3), strides=(1, 1), padding='same',
                  kernel_regularizer=regularizers.l2(weight_decay)))
    md.add(Activation('relu'))
    md.add(BatchNormalization())

    # POOL + Dropout
    md.add(MaxPooling2D(pool_size=(3,3)))
    md.add(Dropout(0.2))

    # CONV_3
    md.add(Conv2D(64, kernel_size=(3, 3), strides=(1, 1), padding='same',
                  kernel_regularizer=regularizers.l2(weight_decay)))
    md.add(Activation('relu'))
    md.add(BatchNormalization())

    # CONV_4
    md.add(Conv2D(64, kernel_size=(3, 3), strides=(1, 1), padding='same',
                  kernel_regularizer=regularizers.l2(weight_decay)))
    md.add(Activation('relu'))
    md.add(BatchNormalization())

    # POOL + Dropout
    md.add(MaxPooling2D(pool_size=(3,3)))
    md.add(Dropout(0.3))

    # CONV_5
    md.add(Conv2D(128, kernel_size=(3, 3), strides=(1, 1), padding='same',
                  kernel_regularizer=regularizers.l2(weight_decay)))
    md.add(Activation('relu'))
    md.add(BatchNormalization())

    # CONV_6
    md.add(Conv2D(128, kernel_size=(3, 3), strides=(1, 1), padding='same',
                  kernel_regularizer=regularizers.l2(weight_decay)))
    md.add(Activation('relu'))
    md.add(BatchNormalization())

    # POOL + Dropout
    md.add(MaxPooling2D(pool_size=(2,2)))
    md.add(Dropout(0.4))

    # FC_7
    md.add(Flatten())
    md.add(Dense(2, activation='softmax'))

    return md

In [ ]:
model = get_model()
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=["accuracy"])

In [ ]:
history = model.fit(ds_train, validation_data=ds_test, epochs=1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epoch_array = range(1, len(history.history['loss']) + 1)
loss_array = history.history['loss']
acc_array = history.history['accuracy']

max_loss = max(loss_array)
loss_array = np.array(loss_array) / max_loss

plt.plot(epoch_array, loss_array, label="loss")
plt.plot(epoch_array, acc_array, label="acc")
plt.xlabel("epochs")
plt.title("total loss and accuracy")

plt.legend()
plt.show()

In [ ]:
epoch_array = range(1, len(history.history['loss']) + 1)
val_acc_array = history.history['val_accuracy']
acc_array = history.history['accuracy']

plt.plot(epoch_array, acc_array, label="acc")
plt.plot(epoch_array, val_acc_array, label="val acc")
plt.xlabel("epochs")
plt.title("val accuracy & accuracy")

plt.legend()
plt.show()

In [ ]:
epoch_array = range(1, len(history.history['loss']) + 1)
val_acc_array = history.history['val_loss']
acc_array = history.history['loss']

plt.plot(epoch_array, acc_array, label="loss")
plt.plot(epoch_array, val_acc_array, label="val loss")
plt.xlabel("epochs")
plt.title("val loss & loss")

plt.legend()
plt.show()

In [ ]:
results = model.evaluate(ds_test)
print(results)

In [ ]:
from matplotlib import rcParams
from matplotlib import pyplot as plt

rcParams["figure.figsize"] = [10, 10]
rcParams['xtick.labelbottom'] = False

In [ ]:
# finally visualize it
x_test = np.concatenate([x for x, y in ds_test], axis=0)
y_test = np.concatenate([y for x, y in ds_test], axis=0)

test_pred = model.predict(x_test)

for idx, elem in enumerate(ds_test.take(25)):
    pred_idx = np.argmax(test_pred[idx])
    true_idx = np.argmax(y_test[idx])
    plt.subplot(5, 5, idx + 1, title=(class_name[pred_idx] + "(" + class_name[true_idx] + ")"))
    plt.imshow(elem[0][0].numpy())

In [ ]:
from matplotlib import pyplot

fig = pyplot.figure(figsize=(20, 8))

for idx, elem in enumerate(ds_test.take(32)):
    ax = fig.add_subplot(4, 8, idx + 1, xticks=[], yticks=[])
    ax.imshow(elem[0][0].numpy())
    pred_idx = np.argmax(test_pred[idx])
    true_idx = np.argmax(y_test[idx])
    ax.set_title("{} ({})".format(class_name[pred_idx], class_name[true_idx]),
                 color=("green" if pred_idx == true_idx else "red"))
pyplot.show()